In [1]:
!pip install -q tqdm

import google.genai as genai
from google.colab import userdata
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
import random
import numpy as np
import re
import time
from tqdm.auto import tqdm
import time
import re

In [2]:
def back_translate(text, client, model="gemini-2.5-flash", delay_seconds=1):
    """
    Performs back-translation on a single text string using the Gemini API.
    (English -> German -> English)

    Includes print statements for success and failure.

    Returns the augmented text, or None if the API fails or text is unchanged.
    """
    try:
        # 1. Translate English to German
        prompt_to_ger = f"Translate the following text from English to German. Output the translated text only:\n{text}"
        response_ger = client.models.generate_content(
            model=model,
            contents=prompt_to_ger
        )
        german_text = response_ger.text

        # Add a delay to respect API rate limits
        time.sleep(delay_seconds)

        # 2. Translate German back to English
        prompt_to_eng = f"Translate the following text from German to English. Output the translated text only:\n{german_text}"
        response_eng = client.models.generate_content(
            model=model,
            contents=prompt_to_eng
        )
        back_translated_text = response_eng.text

        # 3. Clean and check the result
        back_translated_text = re.sub(r'\s+', ' ', back_translated_text).strip()

        # 4. Check for failure (empty string or identical text)
        if not back_translated_text or back_translated_text.lower() == text.lower():
            print("--- API call warning: Back-translation resulted in identical or empty text. Skipping.")
            return None

        # 5. If we reach this point, the call was successful
        print("--- API call SUCCESS: Text augmented.")
        return back_translated_text

    except Exception as e:
        # 6. This catches any API-level errors (e.g., connection, invalid key)
        print(f"--- API call FAILED: {e}. Skipping this item.")
        time.sleep(delay_seconds * 2) # Sleep longer on an error
        return None

In [3]:
def get_augmented_train_set_backtranslation(train_ds, client, seed=42):
    """
    Takes the original train dataset and returns a new, augmented
    dataset using back-translation.
    """
    random.seed(seed)
    np.random.seed(seed)

    # 1. Add 'is_augmented' tracking column to the original data
    def add_tracking_col(example):
        example['is_augmented'] = False
        return example
    train_ds = train_ds.map(add_tracking_col)

    # 2. Define classes and percentages to augment
    classes_to_augment = {
        "Ambivalent": 0.25,
        "Clear Reply": 0.25,
        "Clear Non-Reply": 0.25
    }

    new_rows_list = []
    print("Starting back-translation augmentation process...")

    # 3. Loop over each class to augment
    for label, percentage in classes_to_augment.items():
        print(f"\n--- Augmenting class: {label} ---")

        # Filter for the specific class
        class_subset = train_ds.filter(lambda x: x['clarity_label'] == label)
        num_original = len(class_subset)
        num_to_generate = int(num_original * percentage)

        if num_to_generate == 0:
            print(f"Skipping class '{label}', 0 items to generate.")
            continue

        print(f"Original rows: {num_original}. Attempting to generate {num_to_generate} new rows.")

        # Randomly select indices to augment
        indices_to_augment = random.choices(range(num_original), k=num_to_generate)

        # 4. Call the API for each selected text
        # We use tqdm for a progress bar, as this will be slow.
        print("Calling Back-Translation API (this will take time)...")

        successful_augmentations = 0
        for i in tqdm(indices_to_augment):
            original_row = class_subset[i]
            text_to_augment = original_row['interview_answer']

            # Call our new helper function
            augmented_text = back_translate(text_to_augment, client)

            # If augmentation was successful and created a new text:
            if augmented_text:
                new_row = original_row.copy()
                new_row['interview_answer'] = augmented_text
                new_row['is_augmented'] = True
                new_rows_list.append(new_row)
                successful_augmentations += 1

        print(f"Successfully generated {successful_augmentations} new rows for class '{label}'.")

    print("\n--- Augmentation Complete ---")

    # 5. Create a new Dataset from the augmented rows
    if not new_rows_list:
        print("No new rows were generated. Returning original dataset.")
        return train_ds

    new_rows_ds = Dataset.from_list(new_rows_list, features=train_ds.features)

    # 6. Combine original data with new data
    augmented_train_ds = concatenate_datasets([train_ds, new_rows_ds])

    print(f"Original train size: {len(train_ds)}")
    print(f"New augmented rows: {len(new_rows_ds)}")
    print(f"Final augmented train size: {len(augmented_train_ds)}")

    return augmented_train_ds

In [5]:
# --- 1. Initialize Gemini Client ---
print("Initializing Gemini API Client...")
try:
    # Make sure to add your API key to Colab secrets!
    # Go to "View" -> "Secrets" and add a new secret named "GEMINI_API_KEY"
    api_key = userdata.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError("GEMINI_API_KEY not found in Colab secrets.")

    client = genai.Client(api_key=api_key)
    print("Gemini client initialized successfully.")

except Exception as e:
    print(f"Error initializing Gemini client: {e}")
    print("Please ensure you have added your 'GEMINI_API_KEY' to Colab Secrets (View -> Secrets).")
    # Stop execution if client fails
    raise

# --- 2. Load Original Dataset ---
print("\nLoading original dataset from Hugging Face...")
original_dataset = load_dataset("ailsntua/QEvasion")

# --- 3. Run Augmentation ---
print("\nAugmenting the 'train' split using Back-Translation...")
augmented_train_split = get_augmented_train_set_backtranslation(
    original_dataset['train'],
    client=client
)

# --- 4. Ensure Schema Consistency ---
# (This logic is copied from your 'del.ipynb' and is essential)
print("\nAdding 'is_augmented' column to the 'test' split for schema consistency...")
def add_tracking_col_test(example):
    example['is_augmented'] = False
    return example
original_test_split = original_dataset['test'].map(add_tracking_col_test)

# --- 5. Create and Save Final Dataset ---
print("\nCreating new DatasetDict...")
final_dataset = DatasetDict({
    'train': augmented_train_split,
    'test': original_test_split  # Use the modified test split
})

output_dir = "./backtranslation_augmented_qevasion"
print(f"Saving final dataset to {output_dir}...")
final_dataset.save_to_disk(output_dir)

print("\n--- Done! ---")
print(f"You can now load this dataset for training using:\nload_from_disk('{output_dir}')")

Initializing Gemini API Client...
Gemini client initialized successfully.

Loading original dataset from Hugging Face...

Augmenting the 'train' split using Back-Translation...
Starting back-translation augmentation process...

--- Augmenting class: Ambivalent ---
Original rows: 2040. Attempting to generate 510 new rows.
Calling Back-Translation API (this will take time)...


  0%|          | 0/510 [00:00<?, ?it/s]

--- API call SUCCESS: Text augmented.


KeyboardInterrupt: 

In [6]:
from datasets import load_from_disk
from collections import Counter

# Load the new dataset from disk
reloaded_dataset = load_from_disk("./backtranslation_augmented_qevasion")
print("Reloaded dataset:")
print(reloaded_dataset)

print("\nOriginal vs. Augmented row counts in 'train' split:")
counts = Counter(reloaded_dataset['train']['is_augmented'])
print(f"  Original rows (is_augmented=False): {counts[False]}")
print(f"  New rows (is_augmented=True):      {counts[True]}")

print("\nVerifying 'test' split:")
print(f"  Test rows: {len(reloaded_dataset['test'])}")
print(f"  Test schema: {reloaded_dataset['test'].features}")

Reloaded dataset:
DatasetDict({
    train: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label', 'is_augmented'],
        num_rows: 4273
    })
    test: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label', 'is_augmented'],
        num_rows: 308
    })
})

Original vs. Augmented row counts in 'train' split:
  Original rows (is_augmented=False): 3448
  New rows (is_augmented=True):      825

Verifying 'test' split:
  Test rows: 

In [7]:
!zip -r /content/dataaug-bt-v1.zip /content/backtranslation_augmented_qevasion
# change sample_data.zip to your desired download name Ex: nothing.zip
# change sample_data to your desired download folder name Ex: ner_data


  adding: content/backtranslation_augmented_qevasion/ (stored 0%)
  adding: content/backtranslation_augmented_qevasion/test/ (stored 0%)
  adding: content/backtranslation_augmented_qevasion/test/data-00000-of-00001.arrow (deflated 70%)
  adding: content/backtranslation_augmented_qevasion/test/dataset_info.json (deflated 78%)
  adding: content/backtranslation_augmented_qevasion/test/state.json (deflated 38%)
  adding: content/backtranslation_augmented_qevasion/train/ (stored 0%)
  adding: content/backtranslation_augmented_qevasion/train/data-00000-of-00001.arrow (deflated 81%)
  adding: content/backtranslation_augmented_qevasion/train/dataset_info.json (deflated 84%)
  adding: content/backtranslation_augmented_qevasion/train/state.json (deflated 55%)
  adding: content/backtranslation_augmented_qevasion/dataset_dict.json (stored 0%)
